# HOMER × Pagani 2026 — cross-species autism subtype validation

This notebook walks through HOMER's third-party validation against [Pagani et al. 2026, *Nat Neurosci*](https://www.nature.com/articles/s41593-026-02287-z) ("Autism subtypes identified using cross-species functional connectivity analyses").

Pagani et al. cluster 20 mouse autism models into hyper-connected and hypo-connected FC subtypes, recover the same two subtypes in 1,029 human ASD subjects from ABIDE, and find that each subtype carries a distinct gene/pathway signature (synaptic for hypo, immune for hyper). Their cross-species bridge is **name-based**: they assume mouse "Somatomotor" maps to human "Somatomotor", etc.

**HOMER provides an independent quantitative cross-species coupling π** (1864 mouse × 2094 human parcels) that knows nothing about Pagani's data. We use it to test the paper's four claims:

1. **Claim 1**: Subjects form two FC subtypes recoverable in both species
2. **Claim 2 (scaffolding)**: A name-based mouse↔human network bridge
3. **Claim 3**: Subtype FC perturbation patterns recur cross-species in matching anatomical locations
4. **Claim 4**: The gene/pathway signature recurs cross-species

We test each claim with HOMER and report what we found.


## Setup

Load HOMER's coupling matrix π and the Pagani 2026 supplementary data.

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'experiments' / 'autism_subtypes'))

from homer.data import load_cached

# HOMER coupling (production + all packs)
pi = np.load(ROOT / 'outputs/coupling/pi_fc_plus_SC_with_all_packs.npy')
print(f'HOMER π: {pi.shape}, total mass: {pi.sum():.4f}')

# Mouse + human parcel metadata
M, _ = load_cached('mouse', cache_dir=str(ROOT / 'outputs/anndata'))
H, _ = load_cached('human', cache_dir=str(ROOT / 'outputs/anndata'))
print(f'Mouse: {len(M.var)} parcels   Human: {len(H.var)} parcels')


## Hypothesis 1 — Does Pagani's name-based bridge have biological substance?

The first thing to check is the scaffolding. If HOMER's π preferentially routes mass between like-named networks across species, then Pagani's name-bridge has biological substance.

We aggregate π to a 13×9 mouse-network × human-network matrix and ask: for each canonical mouse↔human pair (e.g. Salience → Salience, Visual → Visual), is the diagonal cell the argmax of its row?


In [ ]:
from importlib import import_module
nc = import_module('01_network_crossvalidation')

# Aggregated-network bridge result (computed by the validation script)
result1 = json.loads((ROOT / 'outputs/logs/autism_subtypes_network_crossval.json').read_text())
pair_scores = result1['target_pair_scores']

print(f"\n{'Pair':<32} | {'HOMER mass on target':>22s} | {'Null':>10s} | {'Ratio':>7s} | argmax?")
print("-" * 95)
for p in pair_scores:
    star = '★' if p.get('is_argmax_diagonal') else ' '
    pair = f"{p['mouse_net']} → {p['human_net']}"
    if 'row_norm_mass' in p:
        print(f"  {pair:<30s} | {p['row_norm_mass']*100:>20.1f}% | {p['expected_mass_null']*100:>9.1f}% | "
              f"{p['ratio_over_null']:>6.2f}× | {star}")

import numpy as _np
n_diag = sum(p.get('is_argmax_diagonal', False) for p in pair_scores)
mean_ratio = _np.mean([p['ratio_over_null'] for p in pair_scores])
print(f"\nResult: {n_diag}/8 canonical pairs are diagonal-argmax "
      f"(mean mass-ratio over a permuted-π null = {mean_ratio:.2f}×).")

# The decisive test: a FAIR spatial null. Spin the mouse network labels on the
# cortical sphere (preserving spatial autocorrelation) and re-route through the
# REAL π, rather than shuffling π's rows (which destroys autocorrelation and is
# far too lenient for smooth maps).
fair = json.loads((ROOT / 'outputs/logs/fair_nulls_discrete.json').read_text())
print(f"\nFair mouse-spin null:  {fair['network_bridge_observed_diag']} diagonal hits observed "
      f"vs {fair['network_bridge_spin_null_mean']:.2f} expected under spin  "
      f"→  spin p = {fair['network_bridge_spin_p']:.3f}  (SURVIVES).")

**Interpretation — Pagani's bridge has real biological substance, and it survives a fair null.** **4 of 8** canonical pairs are diagonal-argmax — **SomatoMotor, DMN, Salience, and Subcortical** route their largest share of mass to their like-named human network. The 4 that miss the argmax are atlas-definition artefacts, not HOMER disagreeing with biology: *Auditory* actually carries the **most** on-target mass relative to null (5.3×) but its single largest cell lands on the adjacent SomatoMotor parcels; *Visual* (Schaefer-17 "Visual" is V1-like only, so higher-order mouse visual maps to DorsAtten); *HC/Limbic*; and *BF/Olfactory* (no clean cortical Schaefer label → routes to Subcortical).

Crucially, this is HOMER's **strong mode**: the diagonal structure **survives a spatial-autocorrelation-preserving spin null** (4 observed vs 1.78 expected, spin p = 0.026). It is not an artefact of the lenient permuted-π null, and it survives stripping HOMER's anchor packs and zeroing the xyz prior — so no single component drives it. *(Discrete region/network correspondence is exactly where HOMER's coupling is trustworthy — see the synthesis and the spin-null experiments in `experiments/spatial_null_check/`.)*

## Hypothesis 2 — Does HOMER's π reproduce Pagani's per-subtype spatial pattern (claim 3) WITHOUT the name-bridge?

This is the headline test. If HOMER's translation of the mouse subtype perturbation matrix through π predicts the human subtype perturbation matrix without the name-bridge, that's independent quantitative replication of Pagani's claim 3.

Method: build a translation operator T (9 × 8) where T[mi, hj] = P(human-net hj | mouse-net mi), derived by row-normalising aggregated π. Predicted human Δ-matrix = Tᵀ · Δ_mouse · T, where Δ_mouse = (hyper − hypo) intensity matrix from ED Fig 1. Compare against observed human Δ (Fig 4e hyper − hypo) over the 36 upper-triangle elements.


In [ ]:
result2 = json.loads((ROOT / 'outputs/logs/autism_subtypes_full_matrix.json').read_text())
print(f"  Pearson r:        {result2['pearson_r']:+.3f}")
print(f"  Analytical p:     {result2['pearson_p_analytical']:.4f}")
print(f"  Empirical p:      {result2['null']['pearson_empirical_p']:.3f} (200 permuted-π trials)")
print(f"  Null mean:        {result2['null']['pearson_mean']:+.3f}")
print(f"  Null 95% CI:      ({result2['null']['pearson_ci95'][0]:+.3f}, "
      f"{result2['null']['pearson_ci95'][1]:+.3f})")
print(f"  Same-sign cells:  {result2['same_sign_fraction']*36:.0f}/36")

# Visualise predicted vs observed scatter
obs = np.array(result2['delta_human_observed_flat'])
pred = np.array(result2['delta_human_predicted_flat'])
fig, ax = plt.subplots(figsize=(7, 5))
same_sign = np.sign(pred) == np.sign(obs)
ax.scatter(obs[same_sign], pred[same_sign], c='#2a9d8f', s=60, alpha=0.8, label='same sign')
ax.scatter(obs[~same_sign], pred[~same_sign], c='#e76f51', s=60, alpha=0.8, label='opposite sign')
z = np.polyfit(obs, pred, 1); xs = np.linspace(obs.min(), obs.max(), 50)
ax.plot(xs, z[0]*xs + z[1], 'k--', linewidth=0.8, label=f'fit (slope={z[0]:+.2f})')
ax.axhline(0, color='black', linewidth=0.5); ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Observed Δ (Pagani Fig 4e, hyper − hypo)')
ax.set_ylabel('Predicted Δ via HOMER π')
ax.set_title(f"r={result2['pearson_r']:+.3f}, p={result2['pearson_p_analytical']:.4f}, emp p<{1/200:.3f}")
ax.legend()
plt.tight_layout(); plt.show()


**Interpretation — partial, and it does NOT survive a fair spatial null.** HOMER's π, fit without any access to Pagani's data, recovers the *dominant* cross-species signal: the largest observed entry — Subcortical–Subcortical Δ ≈ +33 in human — is also HOMER's largest positive prediction, and 22/36 entries (61%) agree in sign (Pearson r = +0.55, analytical p = 0.0005; empirical p = 0.000 vs the permuted-π null).

**But this is a continuous-map correlation — HOMER's weak mode — and it fails the fair test.** (1) It is leverage-driven: removing the single Subcortical–Subcortical point drops Pearson to 0.34, and the rank correlation is non-significant (Spearman ρ = +0.23, p = 0.18). (2) Under a **fair spin null** (spin the mouse input, route through the real π) the correlation **does not survive (spin p = 0.19)** — the permuted-π null was simply too lenient for a smooth map. So HOMER recovers the dominant Δ entry but does **not** reproduce the joint network-pair structure inferentially. (An earlier version reported r = +0.601, ρ = +0.643 from a pre-v2 pipeline; those do not reproduce — see `_audit/FINDINGS_LOG.md` F-007/F-027.)

## Hypothesis 3 — Does HOMER produce an individual-subject ASD classifier?

A sharper version of Pagani's claim 1: can HOMER's translated subtype template distinguish ASD subjects from controls at the individual level, and does it recover Pagani's hyper/hypo bimodal split *within* ASD?

Pipeline: fetched 871 ABIDE-pcp subjects (CPAC, AAL-116, ~24 sites). Per-subject `mean(FC)` per parcel as a perturbation feature, site-matched against control means, mapped to HOMER's 2,094 parcels by nearest centroid, scored by dot-product against the z-scored HOMER (hyper − hypo) template.


In [ ]:
result4 = json.loads((ROOT / 'outputs/logs/autism_subtypes_abide.json').read_text())
print(f"  n valid subjects: {result4['n_valid']} ({result4['n_asd']} ASD + {result4['n_control']} control)")
print(f"\nUsing signed-FC subject feature (audit-corrected):")
sig = result4['by_feature']['signed']
print(f"  ASD score mean:   {sig['asd_mean']:+.2e}  (sd {sig['asd_sd']:.2e})")
print(f"  CTRL score mean:  {sig['ctrl_mean']:+.2e}  (sd {sig['ctrl_sd']:.2e})")
print(f"  Mann-Whitney p:   {sig['p']:.4f}")
print(f"  Cliff's δ:        {sig['cliffs']:+.3f}  (negative = ASD < CTRL on hyper-hypo template)")

print(f"\nWithin-ASD GMM bimodality:")
gmm = result4['gmm_bimodality']
print(f"  Δ BIC (2-comp − 1-comp): {gmm['delta_bic_2_minus_1']:+.1f}  "
      f"({'2-comp preferred' if gmm['two_comp_preferred_bic'] else '1-comp preferred'})")


**Interpretation — null at the subject level.** Re-run end-to-end under the current v2 pipeline and the recommended coupling `pi_fc_plus_SC_with_all_packs.npy`, HOMER's (hyper − hypo) template does **not** distinguish ASD from controls: signed-FC feature p = 0.96 (Cliff's δ = +0.002), `mean(|FC|)` feature p = 0.64 (δ = +0.019) — both non-significant with negligible, sign-inconsistent effect sizes.

Within ASD the distribution is unimodal (GMM 1-component preferred, ΔBIC = +17.8). So HOMER does NOT recover Pagani's hyper/hypo split as a within-ASD subject-level classifier, and shows no consistent ASD-vs-control shift at the individual level.

**Provenance:** an interim (pre-v2) log had shown the signed feature at p = 0.042; that did not survive the pipeline rebuild. HOMER's cross-species signal lives at the population/network level (Tests 2c, 3), not at single-subject resolution. See `_audit/FINDINGS_LOG.md` F-006.


## Hypothesis 1b — Does HOMER's learned coupling subtype individuals as well as Pagani's name-bridge?

Pagani's *actual* human subtyping (their Methods) is a discrete **classification**: take the mouse "prominent" dysconnectivity regions, map them to human regions **by name**, and label each individual hypo/hyper by ±1 s.d. of regional global connectivity. That mouse→human name-matching is exactly the step HOMER's π can replace with a *learned* coupling.

`04_homer_human_masks.py` routes the mouse prominent regions through π to data-driven human masks; `abide_subtype/05_abide_homer_subtyping.py` then re-runs Pagani's ±1 s.d. classification on ABIDE with the HOMER masks **and** the name-matched masks, head to head. Does the learned coupling subtype *more* people, and how much do the two agree?

In [ ]:
masks = json.loads((ROOT / 'outputs/logs/pagani_homer_human_masks.json').read_text())
print(f"π-derived human masks vs name-matched homologue: argmax agrees for "
      f"{masks['argmax_agree']}/{masks['n_regions']} mouse prominent regions.")
for r in masks['region_routing']:
    print(f"  {r['mouse_region']:<20} → top human nets {r['top_human_nets']}  "
          f"(name-expected {r['name_expected']}; agrees={r['argmax_agrees']})")

sub = json.loads((ROOT / 'outputs/logs/abide_homer_subtyping.json').read_text())
h, n = sub['results']['homer'], sub['results']['name']
print(f"\nABIDE re-subtyping (Pagani's ±1 s.d. rule, {h['n_total']} ASD):")
print(f"  HOMER π-masks : {h['pct_subtyped']:>5.1f}% subtyped  (hypo {h['n_hypo']}, hyper {h['n_hyper']})")
print(f"  name-matched  : {n['pct_subtyped']:>5.1f}% subtyped  (hypo {n['n_hypo']}, hyper {n['n_hyper']})")
print(f"  Pagani report : ~{sub['pagani_reference_pct']:.1f}%")
print(f"  label agreement HOMER vs name-matched: {sub['label_agreement']*100:.0f}%")

**Interpretation — HOMER's learned coupling ≈ the name-bridge, and thereby validates it.** The π-derived masks subtype **21.3%** of ASD individuals vs **22.3%** for the name-matched masks (Pagani report ~25%), with **93% label agreement**. So HOMER does *not* subtype more people — it reproduces the name-matched homology with a coupling that never saw Pagani's data, which is independent confirmation that the name-bridge is biologically real (HOMER's strong, discrete mode again). Region-by-region, π agrees with the named homologue for 4/7 prominent regions; the disagreements are meaningful, not errors — e.g. hippocampus→DMN, which the anatomical name-match to "Subcortical" misses.

The corollary: the ~78% left *unsubtyped* is a **hard-threshold** bottleneck, not a mapping failure. So we removed the threshold and gave every individual a continuous position on the HOMER hyper↔hypo axis (`abide_subtype/06_continuous_subtype_score.py`):

In [ ]:
cont = json.loads((ROOT / 'outputs/logs/abide_continuous_subtype.json').read_text())
s = cont['sanity_hard_label_axis_means']; av = cont['asd_vs_ctrl']
print(f"Continuous HOMER hyper↔hypo axis ({cont['n_asd']} ASD, {cont['n_ctrl']} ctrl):")
print(f"  sanity — hard-hyper axis mean {s['hyper']['axis_mean']:+.3f} > hard-hypo {s['hypo']['axis_mean']:+.3f}  "
      f"(axis orders the hard labels correctly ✓)")
print(f"  ASD vs control: Mann-Whitney p = {av['mannwhitney_p']:.2f}  (no diagnostic shift)")
print("  axis vs ADOS symptom severity (Spearman across ALL scored ASD):")
for f, r in cont['ados_doseresponse'].items():
    print(f"    {f:<22} n={r['n']:>4}  ρ={r['spearman_rho']:+.3f}  p={r['p']:.3g}")

**Interpretation — the continuous axis is a clean negative.** The axis is a *valid construct* (it orders the hard labels correctly: hard-hyper +0.110 > hard-hypo +0.025), but it carries **no diagnostic signal** (ASD vs control p = 0.97) and **no severity dose-response** (every ADOS subscale |ρ| ≤ 0.11, all n.s.; the closest, ADOS_SOCIAL ρ = −0.11 p = 0.078, is the "wrong" sign). This is consistent with HOMER's dichotomy — discrete correspondence survives, graded/continuous translation does not — and with the fact Pagani themselves never demonstrated a continuous ADOS dose-response. *Caveat (F-015):* the axis inherits π's uneven human coverage (masks lean Subcortical/Salience/DMN), so the honest claim is "no detectable continuous severity signal under the current coupling," not "the continuum is flat."*

## Hypothesis 4 — Does HOMER validate Pagani's gene/pathway claim (claim 4)?

The most ambitious test. If HOMER's spatial mapping is correct, mouse spatial expression of Pagani's subtype gene sets, translated through π, should predict the human ASD perturbation pattern.

We downloaded parcel-level Allen ISH expression for 1,713 of Pagani's 6,415 implicated genes (27% Allen coverage), built per-parcel mouse spatial scores, translated through π, and correlated the predicted human Δ against Pagani's observed Δ.

Initial reading looks clean. Then the cross-disease specificity check broke the autism-specific story.


In [ ]:
# First the bootstrap result on autism genes
result3 = json.loads((ROOT / 'outputs/logs/autism_subtypes_gene_diagnostics.json').read_text())
boot = result3['bootstrap']
print(f"  Bootstrap r (autism, n_genes={result3['n_genes']}): "
      f"mean={boot['mean_r']:+.3f}, 95% CI ({boot['ci95'][0]:+.3f}, {boot['ci95'][1]:+.3f})")
print(f"  Fraction positive:    {boot['pct_r_positive']*100:.1f}%")
print(f"  Fraction r > 0.3:     {boot['pct_r_above_0_3']*100:.1f}%")
print(f"  → autism gene-spatial pattern translates stably through π")

# Now cross-disease
print(f"\nCross-disease specificity check (MOESM5 other conditions):")
cd = json.loads((ROOT / 'outputs/logs/autism_subtypes_cross_disease.json').read_text())
print(f"  {'Condition':<22s} | {'n genes':>7s} | {'Bootstrap r':>12s} | {'95% CI':>20s}")
print("  " + "-" * 70)
for cond, r in sorted(cd['results'].items(), key=lambda x: -x[1].get('boot_mean', -99)):
    if r.get('skipped'):
        print(f"  {cond:<22s} | {r['n_overlap']:>7d} | (too few — skipped)")
    else:
        print(f"  {cond:<22s} | {r['n_overlap']:>7d} | {r['boot_mean']:>+12.3f} | "
              f"({r['boot_ci95'][0]:+.3f}, {r['boot_ci95'][1]:+.3f})")


**Interpretation — Pagani's claim 4 is partially supported, but NOT in the autism-specific way they frame it.**

HOMER's translation of the autism gene set produces a bootstrap-stable cross-species correlation (r = +0.43, 95% CI doesn't cross zero). Initially this looked like clean independent validation.

**But the cross-disease specificity check broke the autism-specific story.** ADHD genes give r = +0.45 (beats autism!). Schizophrenia genes give r = +0.43 (ties autism). Bipolar genes give r = +0.41. All overlapping CIs. HOMER's translation of *any* brain-disorder gene set hits Pagani's observed human ASD pattern at the same r ≈ +0.4.

The honest reformulation: HOMER captures **shared spatial geometry of brain gene expression that overlaps with where psychiatric perturbation effects live in the human brain** — but it does not distinguish autism's gene set from other psychiatric conditions.


## Synthesis — the dichotomy: where HOMER's signal lives

The Pagani validation resolves into one clean rule, confirmed against **fair, spatial-autocorrelation-preserving nulls**: **HOMER reliably recovers *discrete* cross-species correspondence, but does *not* translate *continuous / graded* structure.**

| HOMER contribution | Granularity | Outcome | Fair-null verdict |
|---|---|:---|:---|
| Name-based network bridge has substance | 8 networks | **Yes** — 4/8 diagonal-argmax | **Survives spin** (p = 0.026) |
| Learned coupling re-subtypes individuals like the name-bridge | Individual (discrete) | **Yes** — 21.3% vs 22.3%, 93% agreement | (validates the name-match) |
| Reproduces per-subtype FC Δ-matrix across species | 36 network-pair elements | **Partial** — r = +0.55 (leverage-driven) | **Fails spin** (p = 0.19) |
| Continuous hyper↔hypo severity axis | Individual (continuous) | **No** — ASD≈ctrl p = 0.97; ADOS all n.s. | (continuous = weak mode) |
| Distinguishes ASD vs control at subject level | Individual | **No** — null (p = 0.96, δ ≈ 0) | — |
| Autism-*specific* gene/pathway claim | Disease specificity | **No** — same r for ADHD/SCZ/bipolar | — |
| Shared brain-disorder spatial geometry | Population spatial | **Yes** — stable r ≈ +0.4 across psych conditions | — |

**The honest, outward-facing story.** HOMER's coupling is trustworthy exactly where the question is *discrete correspondence*: it validates Pagani's name-based bridge (survives a spin null) and reproduces their individual-level subtyping (93% agreement) with a coupling that never saw their data. It is *not* trustworthy for *continuous-map* translation: the Δ-matrix correlation and the continuous severity axis are both null under fair tests. Reporting both — the strong discrete result and the honest continuous null — is what makes the discrete result credible.

For full details: [`docs/03_results.md`](../docs/03_results.md) — "Independent validation against Pagani 2026"; spin-null experiments in [`experiments/spatial_null_check/`](../experiments/spatial_null_check/).

_Related: notebook `09_pagani_per_model_translation.ipynb` covers the per-model / mouse-side translation._